# SD 인페인팅 + LoRA — VisA PCB1–3 K장 생성

**런타임 → 런타임 유형 변경 → GPU(T4 또는 A100)** 를 고르고 **런타임 → 모두 실행**을 누릅니다. Drive 마운트를 승인하면 끝까지 돕니다.

| 단계 | 내용 | T4 예상 시간 |
|---|---|---|
| 학습 | 조건 (K, seed)마다 PCB별 LoRA 1개, K장으로만 학습 | LoRA 1개당 수십 분 (첫 실행에서 실제 시간이 출력됨) |
| 생성 | PCB 3 × 종류 4 × 40장 = 480장, 결함 위치는 copy-paste와 같은 규칙(기판 정렬 + 종류별 jitter) | 조건당 10분 안팎 |

- 기본은 **K=1 파일럿**입니다. 결과를 보고 첫 코드 셀의 `K_VALUES = (1, 2, 4)`로 바꿔 다시 **모두 실행**하면 됩니다.
- LoRA와 결과는 Drive `MyDrive/DL_TP/runs/sd_lora/`에 저장됩니다. **세션이 끊겨도 다시 모두 실행하면 끝난 것은 건너뛰고 이어서** 합니다.
- 결과 형식은 copy-paste·cTransGAN과 같습니다(`Dataset/generated/sd_lora/K{k}_seed{s}/`, 원본 해상도 JPEG q75, 종류 ID 마스크, manifest). 생성이 끝나면 누수 검사가 자동으로 돕니다.
- 코드 설명: `Augmentation/sd_lora/README.md`

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
from google.colab import drive

REPO_URL = 'https://github.com/Junho-YEO-maker/26_2_DeepLearning.git'
BRANCH = 'main'                       # 특정 커밋으로 고정하려면 40자리 커밋 해시를 넣는다
REPO = Path('/content/26_2_DeepLearning')
AUG = REPO / 'Augmentation'
DRIVE = Path('/content/drive/MyDrive/DL_TP')
RUN = DRIVE / 'runs/sd_lora'
K_VALUES = (1,)                       # 파일럿 후 (1, 2, 4)
SPLIT_SEED = 0
PER_TYPE = 40
STEPS = 1000                          # LoRA 학습 step

drive.mount('/content/drive')
DRIVE.mkdir(parents=True, exist_ok=True)      # 결과 저장 폴더 (없으면 만든다)
subprocess.run(['nvidia-smi'], check=True)
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
target = BRANCH if len(BRANCH) == 40 else f'origin/{BRANCH}'
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', target], check=True)
COMMIT = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('코드 커밋:', COMMIT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'diffusers>=0.30', 'peft>=0.12', 'accelerate>=0.33'],
               check=True)
# Colab에 미리 깔린 torchao(0.10)가 너무 오래돼서 최신 peft가 LoRA를 만들다 멈춘다. 우리는 torchao를 안 쓰므로 지운다.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)
RUN.mkdir(parents=True, exist_ok=True)
with (RUN / 'commits.txt').open('a') as f:
    f.write(COMMIT + '\n')
os.chdir(AUG)
sys.path.insert(0, str(AUG))


def run(*args):
    # 하위 프로세스 출력을 셀에 그대로 보여 준다 (Colab은 subprocess 출력을 셀에 안 보여 줌)
    proc = subprocess.Popen([sys.executable, '-u', *map(str, args)], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, env=dict(os.environ, PYTHONUNBUFFERED='1'))
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait():
        raise RuntimeError(f'실패 (exit {proc.returncode}): 위 출력의 마지막 오류 메시지를 확인하세요')

### 원본 데이터
Drive에 올려 둔 `VisA_official` 폴더(안에 `VisA_PCB/pcb1, pcb2, pcb3`)를 씁니다.
`VISA_PCB`를 비워 두면 내 드라이브에서 `VisA_PCB` 폴더를 자동으로 찾습니다.
공유받은 폴더라면 Drive에서 그 폴더를 **우클릭 → 정리 → 바로가기 추가 → 내 드라이브**를 먼저 해 두세요(Colab은 내 드라이브만 보입니다).

In [ ]:
# Drive의 VisA_PCB 폴더를 찾아 /content로 복사 (Drive에서 직접 읽으면 느리다)
VISA_PCB = None        # 자동으로 못 찾으면 직접: Path('/content/drive/MyDrive/<폴더>/VisA_official/VisA_PCB')

BOARDS = ('pcb1', 'pcb2', 'pcb3')
TYPES = ('bent', 'melt', 'scratch', 'missing')
LOCAL_DATA = Path('/content/visa_pcb123')

def find_visa_pcb():
    root = Path('/content/drive/MyDrive')
    patterns = ['VisA_PCB', '*/VisA_PCB', '*/*/VisA_PCB', '*/*/*/VisA_PCB']
    hits = [p for pat in patterns for p in root.glob(pat) if (p / 'pcb1/image_anno.csv').is_file()]
    assert hits, ('내 드라이브에서 VisA_PCB 폴더를 못 찾았습니다. 업로드했는지, 공유 폴더면 바로가기를 추가했는지 확인하거나 '
                  'VISA_PCB에 경로를 직접 넣으세요.')
    if len(hits) > 1:
        print('VisA_PCB 폴더가 여러 개입니다. 첫 번째를 씁니다:', *hits, sep='\n  ')
    return hits[0]

if not all((LOCAL_DATA / p / 'image_anno.csv').is_file() for p in BOARDS):
    src = Path(VISA_PCB) if VISA_PCB else find_visa_pcb()
    print('Drive 원본:', src, '→ 복사 중 (수 분 걸릴 수 있음)')
    for p in BOARDS:
        shutil.copytree(src / p, LOCAL_DATA / p, dirs_exist_ok=True)
for p in BOARDS:
    n = len(list((LOCAL_DATA / p / 'Data/Images/Normal').iterdir())) + len(list((LOCAL_DATA / p / 'Data/Images/Anomaly').iterdir()))
    print(p, n, '장')

# 공유 분할표·원본 해시 확인 (쓰는 이미지는 열 때마다 해시를 대조하고, test/val 이미지는 열 수 없게 막혀 있다)
from common.kshot import KShot
for k in K_VALUES:
    KShot(k, SPLIT_SEED, LOCAL_DATA)
print('원본 데이터:', LOCAL_DATA, '| 분할표 확인 완료:', K_VALUES)

### 학습
PCB마다 LoRA 1개. Drive에 이미 있는 LoRA는 건너뜁니다. 출력되는 학습 시간을 보고 나머지 일정을 잡으세요.

In [ ]:
import json
for k in K_VALUES:
    run('-m', 'sd_lora.train', '--k', k, '--seed', SPLIT_SEED, '--steps', STEPS,
        '--data-root', LOCAL_DATA, '--runs-dir', RUN / 'lora')
    for pcb in BOARDS:
        info = json.loads((RUN / f'lora/K{k}_seed{SPLIT_SEED}/{pcb}/train.json').read_text())
        print(f'K{k} {pcb}: {info["minutes"]:.1f}분, 마지막 loss {info["final_loss"]:.4f}, {info["gpu"]}')

### 생성 → Drive 저장
조건마다 480장. Drive에 manifest가 이미 있으면 건너뜁니다.

In [ ]:
from collections import Counter
from common.kshot import check_manifest, read_rows

def publish(k):
    src = REPO / f'Dataset/generated/sd_lora/K{k}_seed{SPLIT_SEED}'
    rows = read_rows(src / 'manifest.csv')
    counts = Counter((r['category'], r['defect_type']) for r in rows)
    assert counts == {(p, t): PER_TYPE for p in BOARDS for t in TYPES}, counts
    print(check_manifest(src / 'manifest.csv'))
    dest = RUN / src.relative_to(REPO)
    shutil.copytree(src, dest, dirs_exist_ok=True, ignore=shutil.ignore_patterns('manifest.csv'))
    shutil.copyfile(src / 'manifest.csv', dest / 'manifest.csv.tmp')
    (dest / 'manifest.csv.tmp').replace(dest / 'manifest.csv')     # manifest는 마지막에: 있으면 완료된 것
    print('Drive 저장 완료:', dest, len(rows), '장')

for k in K_VALUES:
    if (RUN / f'Dataset/generated/sd_lora/K{k}_seed{SPLIT_SEED}/manifest.csv').is_file():
        print(f'[건너뜀] K{k}_seed{SPLIT_SEED} 이미 Drive에 있음')
        continue
    run('-m', 'sd_lora.generate', '--k', k, '--seed', SPLIT_SEED, '--per-type', PER_TYPE,
        '--data-root', LOCAL_DATA, '--runs-dir', RUN / 'lora', '--force')
    publish(k)

### 눈으로 확인
PCB·종류마다 한 장씩 결함 주변을 확대해서 보여 줍니다(빨간 테두리 = 마스크). 결함이 실제로 그려졌는지, 기판의 맞는 자리인지 확인하세요.

In [ ]:
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display

k = K_VALUES[0]
rows = read_rows(RUN / f'Dataset/generated/sd_lora/K{k}_seed{SPLIT_SEED}/manifest.csv')
tiles = []
for pcb in BOARDS:
    for t in TYPES:
        r = next(x for x in rows if x['category'] == pcb and x['defect_type'] == t)
        im = Image.open(RUN / r['image']).convert('RGB')
        m = np.asarray(Image.open(RUN / r['mask'])) > 0
        ys, xs = np.where(m)
        cx, cy = (xs.min() + xs.max()) // 2, (ys.min() + ys.max()) // 2
        half = max(128, (max(np.ptp(xs), np.ptp(ys)) + 60) // 2)
        box = (cx - half, cy - half, cx + half, cy + half)
        crop = im.crop(box)
        ImageDraw.Draw(crop).rectangle([xs.min() - box[0] - 4, ys.min() - box[1] - 4,
                                        xs.max() - box[0] + 4, ys.max() - box[1] + 4], outline='red', width=3)
        crop = crop.resize((256, 256))
        ImageDraw.Draw(crop).text((4, 4), f'{pcb} {t}', fill='yellow')
        tiles.append(crop)
grid = Image.new('RGB', (256 * 4, 256 * 3))
for i, tile in enumerate(tiles):
    grid.paste(tile, ((i % 4) * 256, (i // 4) * 256))
display(grid)

### 팀에 넘기기
- 이미지·마스크: Drive `MyDrive/DL_TP/runs/sd_lora/Dataset/generated/sd_lora/K{k}_seed0/` (폴더 공유 링크를 AD 팀에 전달)
- manifest: 같은 폴더의 `manifest.csv`를 저장소 `Dataset/generated/sd_lora/K{k}_seed0/manifest.csv`로 커밋
- 실행 커밋 기록: `MyDrive/DL_TP/runs/sd_lora/commits.txt`